<img src='https://hammondm.github.io/hltlogo1.png' style="float:right">
Davo Acevedo-Cardona<br>
Fall 2025<br>
Self-Paced Listening Project / Process AFTER PsychoPy Experiment.

## This code is used to process the data AFTER ALL the data is collected in Psychopy:

This code is just to dejunk all the data from Psychopy so it's more easy to understand it.  
All the new data will be in the folder `data_clean`  
NOTE: IT DOES NOT ERASE OR MODIFY ANY VALUE JUST REORGANIZE IT.

#### Before running the code:
- Make sure to close all the CSV files.
- Verify that the "Paths" matches with the ones in the machine.
  
#### Running the code
1. On the top bar check for the option that says: `kernel` 
2. Click on `kernel` option the search for `Restart Kernel and Run All Cells...`
3. A window will open, click the red button: `Restart` 
4. Wait until you see the message "All CSVs processed successfully!" and check the `data_clean` folder.

# Imports

In [ ]:
import os
import pandas as pd
import numpy as np

# Paths

In [ ]:
base_dir = #YOUR_PATH
output_dir = #YOUR_PATH
os.makedirs(output_dir, exist_ok=True)

# Preprocess (DO NOT CHANGE ANYTHING)

In [ ]:
# === Columns to remove ===
cols_to_drop = [
    # psychoPy auto-metadata
    "thisN", "thisTrialN", "thisRepN",
    "trials.thisRepN", "trials.thisTrialN", "trials.thisN", "trials.thisIndex",
    "question_trials.thisRepN", "question_trials.thisTrialN",
    "question_trials.thisN", "question_trials.thisIndex", "spacebar_rt_ms", "Unnamed: 109",
    "space_delay.keys", "space_delay.rt", "space_delay.duration",
    "space_delay_3.keys", "space_delay_3.rt", "space_delay_3.duration",
    "trials_prueba.space_delay.keys", "trials_prueba.space_delay.rt",
    "trials_prueba.space_delay.duration",	"trials.space_delay_3.keys",
    "trials.space_delay_3.rt",	"trials.space_delay_3.duration",

    # unwanted response fields
    "resp_q.keys", "resp_q.rt", "resp_q.duration",
    "question_trials.resp_q.keys", "question_trials.resp_q.rt", "question_trials.resp_q.duration",
    "trials.resp_q.keys", "trials.resp_q.rt", "trials.resp_q.duration", "list_prueba",

    # NEW: Drop keyboard columns from prueba
    "key_prueba.keys", "key_prueba.rt", "key_prueba.duration",
    "resp_q_2.keys", "resp_q_2.rt", "resp_q_2.duration",
    "trials_prueba.thisRepN", "trials_prueba.thisTrialN",
    "trials_prueba.thisN", "trials_prueba.thisIndex",
    "trials_prueba.key_prueba.keys", "trials_prueba.key_prueba.rt",
    "trials_prueba.key_prueba.duration",
    "trials_prueba.resp_q_2.keys", "trials_prueba.resp_q_2.rt",
    "trials_prueba.resp_q_2.duration",

    # metadata
    "expName", "expVersion", "psychopyVersion", "frameRate",
    "expStart", "thisRow.t", "Unnamed: 96", "Unnamed: 120",

    # NEW: columns you asked to drop
    "notes", "session"
]

# === Process CSV files ===
for file in os.listdir(base_dir):
    if not file.endswith(".csv"):
        continue

    print(f"\nProcessing {file}...")
    file_path = os.path.join(base_dir, file)

    df = pd.read_csv(file_path, encoding="utf-8")

    # Remove fully empty rows
    df = df.dropna(how="all")

    df = df.rename(columns={"duration_ms": "audio_length"})

    # Drop unwanted columns
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns], errors="ignore")

    # Clean whitespace
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].astype(str).str.strip()

    # Convert literal "nan" → NaN
    df = df.replace("nan", np.nan)

    # === Remove all PRUEBA rows ===
    if "spl_exp" in df.columns:
        df = df[df["spl_exp"] != "prueba"]

    # === Clean response columns ===
    for col in ["resp_key", "resp_label", "resp_q_rt_ms"]:
        if col in df.columns:
            df[col] = df[col].replace(["[]", ""], np.nan)

    if "resp_label" in df.columns:
        df["resp_label"] = df["resp_label"].replace("Sin respuesta", np.nan)

    if "resp_key" in df.columns:
        df["resp_key"] = df["resp_key"].where(df["resp_key"].isin(["f", "j"]), np.nan)

    # ============================================================
    # === QUESTIONNAIRE DETECTION ================================
    # ============================================================

    # A trial row ALWAYS has a real stimulus:
    # audio_file + segment_number + item_id
    trial_mask = (
        df["audio_file"].notna() &
        df["segment_number"].notna() &
        df["item_id"].notna()
    )

    df_trials = df[trial_mask].copy().reset_index(drop=True)
    df_q = df[~trial_mask].copy().dropna(how="all")

    # Build questionnaire row if present
    if not df_q.empty:
        q_dict = {}
        for col in df_q.columns:
            vals = df_q[col].dropna().tolist()
            q_dict[col] = vals[-1] if vals else np.nan

        q_row = pd.DataFrame([q_dict])

        # Ensure all columns align
        for c in df_trials.columns:
            if c not in q_row.columns:
                q_row[c] = np.nan

        q_row = q_row[df_trials.columns]

        # Final table: questionnaire row, then data starting at row 3
        df_final = pd.concat([q_row, df_trials], ignore_index=True)

    else:
        df_final = df_trials.copy()

    # === Move timing/response columns next to spl_exp ===
    move_cols = [c for c in df_final.columns if any(x in c.lower() for x in [
        "spacebar", "reaction", "resp_key", "resp_label", "resp_q_rt"
    ])]

    if "spl_exp" in df_final.columns:
        cols = list(df_final.columns)
        insert_pos = cols.index("spl_exp") + 1

        for mc in move_cols:
            if mc in cols:
                cols.remove(mc)
                cols.insert(insert_pos, mc)
                insert_pos += 1

        df_final = df_final[cols]

    # === Move "list" column to first position ===
    if "list" in df_final.columns:
        cols = ["list"] + [c for c in df_final.columns if c != "list"]
        df_final = df_final[cols]

    # === Save file ===
    out_path = os.path.join(output_dir, file.replace(".csv", "_clean.csv"))
    df_final.to_csv(out_path, index=False, encoding="utf-8-sig")

    print(f"Saved cleaned file: {out_path}")

# Output

In [ ]:
print("\nAll CSVs processed successfully!")
print("\nCheck the following folder: YOUR_PATH")